# 📝 Lecture 9 Activity Notebook: Text Vectorization
## AA640: Data Analytics and Text Mining
### Bryant University | Prof. Gianluca Brero

---

**IMPORTANT:** *Before starting, save a copy to your Drive via `File > "Save a copy in Drive"`*

### 🎯 Estimated Time: 90 Minutes

#### In-Class Schedule

| Time | Clock | Block |
|------|-------|-------|
| 30 min | 6:30pm | Lecture: From Words to Numbers |
| 20 min | 7:00pm | **Activity 1** — Romeo vs. Juliet |
| 30 min | 7:20pm | Lecture: TF-IDF & N-grams |
| 30 min | 7:50pm | Break + Quiz |
| 20 min | 8:20pm | **Activity 2** — Presidential Debates |
| 30 min | 8:40pm | Open time — questions, finish notebook |

#### After-Class Activities (~30 min)
*Complete these on your own after lecture.*

| Activity | Topic | Time |
|----------|-------|------|
| Activity 3 | Bigram TF-IDF by Party | ~15 min |
| Activity 4 | TF-IDF Word Clouds by Party | ~15 min |

### ⚙️ How to Use This Notebook
1. **Read** each section carefully
2. **Run** the example cells first to see how things work
3. **Complete** the activities marked with 📝
4. **Check** your answers against the expected output

> 💡 **Tip:** If you get stuck, re-read the example cell right above the activity — the pattern is always there!

## Setup: Import Libraries

In [ ]:
# Install wordcloud (needed in Colab)
!pip install wordcloud -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import re
import urllib.request
from collections import Counter

# NLP Tools
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk import bigrams
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk

# Download NLTK Resources
nltk.download('stopwords')
nltk.download('punkt_tab')

print("Setup complete!")

---
## Loading Dataset 1: Romeo & Juliet

For Activity 1, we'll analyze Shakespeare's **Romeo and Juliet**.
We'll extract each character's lines and explore what they talk about.

The text is loaded directly from [Project Gutenberg](https://www.gutenberg.org/).

In [ ]:
# Download Romeo and Juliet from Project Gutenberg
url = 'https://www.gutenberg.org/cache/epub/1513/pg1513.txt'
response = urllib.request.urlopen(url)
full_text = response.read().decode('utf-8')

# Extract the play text (between ACT I and the end marker)
start = full_text.find('ACT I')
end = full_text.find('*** END OF THE PROJECT GUTENBERG')
play_text = full_text[start:end]

print(f'Play text loaded: {len(play_text):,} characters')
print(play_text[:300])

### Extract Lines by Character

In the Gutenberg format, character names appear in uppercase (e.g., `ROMEO.`, `JULIET.`).
We'll extract all lines spoken by Romeo and Juliet.

In [ ]:
def extract_character_lines(text, character):
    """Extract all lines spoken by a character from the play text."""
    lines = []
    capturing = False
    for line in text.split('\n'):
        stripped = line.strip()
        if stripped.startswith(character + '.'):
            capturing = True
            rest = stripped[len(character) + 1:].strip()
            if rest:
                lines.append(rest)
        elif capturing:
            if re.match(r'^[A-Z]{2,}', stripped) and stripped.endswith('.'):
                capturing = False
            elif stripped.startswith('[') or stripped.startswith('Enter') or stripped.startswith('Exit'):
                continue
            elif stripped:
                lines.append(stripped)
    return ' '.join(lines)

romeo_text = extract_character_lines(play_text, 'ROMEO')
juliet_text = extract_character_lines(play_text, 'JULIET')

print(f"Romeo's lines: {len(romeo_text):,} characters")
print(f"Juliet's lines: {len(juliet_text):,} characters")
print()
print('Romeo (first 200 chars):', romeo_text[:200])

### Build a DataFrame and Tokenize

In [ ]:
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
# Create a DataFrame with one row per character

rj_df = pd.DataFrame({
    'Character': ['Romeo', 'Juliet'],
    'Text': [romeo_text, juliet_text]
})

# Clean and tokenize
stop_words = set(stopwords.words('english'))

def clean_text(text):
    """Clean and tokenize: lowercase, remove punctuation, remove stopwords."""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w.isalpha() and w not in stop_words]
    return tokens

rj_df['tokens'] = rj_df['Text'].apply(clean_text)

print('Romeo tokens (first 15):', rj_df.loc[0, 'tokens'][:15])
print('Juliet tokens (first 15):', rj_df.loc[1, 'tokens'][:15])
print()
print(f"Romeo: {len(rj_df.loc[0, 'tokens'])} words")
print(f"Juliet: {len(rj_df.loc[1, 'tokens'])} words")

---
# 📌 IN-CLASS ACTIVITIES

Complete these sections during class time. Each activity has demo cells above it — run the demos first, then work on the 📝 activity.

---

---
## Activity 1: Romeo vs. Juliet — Word Frequencies, Word Clouds & TF-IDF (~20 min)

*Your English professor hands you the full text of Romeo and Juliet and asks: "What does each character **actually talk about**? And what makes their language **different**?"*

---

### Demo: Counting Word Frequencies

Let's count word frequencies across **all** lines (both characters combined).

In [ ]:
# Flatten all tokens into a single list
all_words = [w for tokens in rj_df['tokens'] for w in tokens]

print(f'Total words (after cleaning): {len(all_words):,}')
print()

# Count word frequencies
word_freq = Counter(all_words)

# Convert to DataFrame
word_count_df = pd.DataFrame(word_freq.most_common(100), columns=['Word', 'Count'])

print('Top 10 words:')
print(word_count_df.head(10))

### Demo: Bar Chart of Top Words

In [ ]:
# Plot the top 20 most common words
top_20 = word_count_df.head(20)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_20, y='Word', x='Count', color='steelblue')
plt.title('Top 20 Most Common Words in Romeo and Juliet', fontsize=14)
plt.xlabel('Frequency')
plt.ylabel('Word')
plt.tight_layout()
plt.show()

### Demo: Word Cloud

In [ ]:
# Generate a word cloud from the top 100 words
word_freq_dict = dict(word_count_df.values)

wordcloud = WordCloud(width=800, height=400, background_color='white',
                      max_words=100, random_state=42).generate_from_frequencies(word_freq_dict)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud)
plt.axis('off')
plt.title('Word Cloud: Romeo and Juliet', fontsize=16)
plt.show()

### 📝 Activity 1a: Top Words for Romeo vs. Juliet

Split the data by character and create a **bar chart of the top 15 words** for each.

Steps:
1. Get Romeo's tokens: `rj_df[rj_df['Character'] == 'Romeo']`
2. Flatten and count with `Counter`
3. Create a DataFrame of the top 15 words
4. Plot a bar chart
5. Repeat for Juliet

*Hint: follow the exact same pattern as the demo above, just filter to one character.*

In [ ]:
# Activity 1a: Bar charts of top 15 words for Romeo and Juliet
# Expected output: two bar charts, one per character
# Hint: filter rj_df by character, flatten tokens, count with Counter,
#       make a DataFrame of top 15, plot with sns.barplot

# --- Romeo ---
# YOUR CODE HERE
top15_romeo = rj_df[rj_df['Character'] == 'Romeo']
all_words_romeo = [w for tokens in top15_romeo['tokens'] for w in tokens]
word_freq_romeo = Counter(all_words_romeo)
word_count_df_romeo = pd.DataFrame(word_freq_romeo.most_common(15), columns=['Word', 'Count'])
plt.figure(figsize=(10, 6))
sns.barplot(data=word_count_df_romeo, y='Word', x='Count', color='steelblue')
plt.title('Top 15 Most Common Words in Romeo')
plt.xlabel('Frequency')
plt.ylabel('Word')
plt.tight_layout()
plt.show()


# YOUR CODE HERE
top15_juliet = rj_df[rj_df['Character'] == 'Juliet']
all_words_juliet = [w for tokens in top15_juliet['tokens'] for w in tokens]
word_freq_juliet = Counter(all_words_juliet)
word_count_df_juliet = pd.DataFrame(word_freq_juliet.most_common(15), columns=['Word', 'Count'])
plt.figure(figsize=(10, 6))
sns.barplot(data=word_count_df_juliet, y='Word', x='Count')
plt.title('Top 15 Most Common Words in Juliet')
plt.xlabel('Frequency')
plt.ylabel('Word')
plt.tight_layout()
plt.show()



### 📝 Activity 1b: Word Clouds for Romeo vs. Juliet

Create a **word cloud** for Romeo and another for Juliet using their top 100 words.

Steps:
1. Use the `Counter` you already created for each character in 1a
2. Convert the top 100 words to a dictionary: `dict(Counter.most_common(100))`
3. Generate a `WordCloud` from frequencies
4. Display with `plt.imshow()`

*Hint: follow the word cloud demo pattern above.*

In [ ]:
# Activity 1b: Word clouds for Romeo and Juliet
# Expected output: two word clouds, one per character
# Hint: use the Counter from 1a, convert top 100 to dict,
#       pass to WordCloud(...).generate_from_frequencies(dict)
from collections import Counter
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import re
# --- Romeo Word Cloud ---
# YOUR CODE HERE
top15_romeo = rj_df[rj_df['Character'] == 'Romeo']
all_words_romeo = [w for tokens in top15_romeo['tokens'] for w in tokens]
word_freq_romeo = Counter(all_words_romeo)
word_freq_dict_romeo = dict(word_freq_romeo.most_common(100))
wordcloud_romeo = WordCloud(width=800, height=400, background_color='white',
                      max_words=100, random_state=42).generate_from_frequencies(word_freq_dict_romeo)

plt.figure(figsize=(10, 6))
plt.imshow(wordcloud_romeo, interpolation='bilinear')
plt.axis('off')
plt.title('Romeo Word Cloud')
plt.show()
# --- Juliet Word Cloud ---
# YOUR CODE HERE
top15_juliet = rj_df[rj_df['Character'] == 'Juliet']
all_words_juliet = [w for tokens in top15_juliet['tokens'] for w in tokens]
word_freq_juliet = Counter(all_words_juliet)
word_freq_dict_juliet = dict(word_freq_juliet.most_common(100))
wordcloud_juliet = WordCloud(width=800, height=400, background_color='white',
                      max_words=100, random_state=42).generate_from_frequencies(word_freq_dict_juliet)

plt.figure(figsize=(10, 6))
plt.imshow(wordcloud_juliet, interpolation='bilinear')
plt.axis('off')
plt.title('Juliet Word Cloud')
plt.show()

# Reflection:
# Do the word clouds look similar? What words appear in both?
# Answer: ___Yes, "thou", "thy" and "love" all appear____________


### 📝 Activity 1c: TF-IDF — What Makes Romeo Different from Juliet?

The word clouds probably look similar — both characters talk about love, night, etc. **TF-IDF** can reveal what's *unique* to each character.

Steps:
1. Join each character's tokens back into a string
2. Fit a `TfidfVectorizer` on the two strings (one per character)
3. Get the top 10 TF-IDF words for Romeo and for Juliet
4. Plot side-by-side bar charts

*Hint: each character's joined text becomes one "document" for TF-IDF.*

In [ ]:
# Activity 1c: TF-IDF comparison of Romeo vs. Juliet
# Expected output: two bar charts showing each character most distinctive words

# Step 1: Join tokens into strings (one document per character)
rj_df["joined"] = rj_df["tokens"].apply(lambda x: " ".join(x))
docs = rj_df["joined"].tolist()

# Step 2: Fit TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(docs)
words = vectorizer.get_feature_names_out()

# Step 3: Create a DataFrame of TF-IDF scores (rows=words, columns=characters)
tfidf_df = pd.DataFrame(tfidf_matrix.T.toarray(), index=words, columns=rj_df["Character"])

# Get top 10 TF-IDF words for Romeo
romeo_top_10_tfidf = tfidf_df['Romeo'].sort_values(ascending=False).head(10)

# Get top 10 TF-IDF words for Juliet
juliet_top_10_tfidf = tfidf_df['Juliet'].sort_values(ascending=False).head(10)

# Step 4: Plot side-by-side bar charts
fig, axes = plt.subplots(ncols=2, figsize=(16, 6), sharey=False)

sns.barplot(x=romeo_top_10_tfidf.values, y=romeo_top_10_tfidf.index, ax=axes[0], color='steelblue')
axes[0].set_title('Top 10 TF-IDF Words for Romeo')
axes[0].set_xlabel('TF-IDF Score')
axes[0].set_ylabel('Word')

sns.barplot(x=juliet_top_10_tfidf.values, y=juliet_top_10_tfidf.index, ax=axes[1], color='coral')
axes[1].set_title('Top 10 TF-IDF Words for Juliet')
axes[1].set_xlabel('TF-IDF Score')
axes[1].set_ylabel('Word')

plt.tight_layout()
plt.show()

# Reflection:
# How do these TF-IDF results differ from the word clouds?
# Answer: __Easier to tell which words are used and how frequently by a precise number_____________

---
## Loading Dataset 2: Presidential Debates

For Activity 2, we'll switch to a real-world dataset: transcripts from the **2016 U.S. Presidential Primary Debates**.

**To load the file in Colab:**
1. Click the **folder icon** on the left sidebar.
2. Click the **upload button** (the icon with an upward arrow).
3. Select `primary_debates_cleaned.csv` from your computer.
4. Then run the cell below.

In [ ]:
# Load the debate transcript data
raw_debates = pd.read_csv('primary_debates_cleaned.csv')

print(f"Dataset: {raw_debates.shape[0]} rows, {raw_debates.shape[1]} columns")
print()
raw_debates.head()

### Filter to Candidate Statements

Not everyone who speaks in a debate is a candidate (moderators, audience members, etc.). Let's keep only the actual candidates.

In [ ]:
# List of known presidential candidates
candidates = ['Bush', 'Carson', 'Chafee', 'Christie', 'Clinton', 'Cruz', 'Fiorina',
              'Gilmore', 'Graham', 'Huckabee', 'Jindal', 'Kasich', "O'Malley",
              'Pataki', 'Paul', 'Perry', 'Rubio', 'Sanders', 'Santorum', 'Trump',
              'Walker', 'Webb']

# Filter to candidates only
debates = raw_debates[raw_debates['Speaker'].isin(candidates)].copy()

# Standardize party labels
debates['Party'] = debates['Party'].replace('Republican Undercard', 'Republican')

# Keep only the columns we need
debate_text = debates[['Party', 'Text']].copy()

print(f"Candidate statements: {len(debate_text)}")
print()
print(debate_text['Party'].value_counts())

### Clean and Tokenize the Text

Before we can count words, we need to clean the text. This function:
1. Converts to lowercase
2. Removes punctuation
3. Splits into individual words (tokens)
4. Removes stopwords ("the", "is", "and", etc.)

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    """Clean and tokenize text: lowercase, remove punctuation, remove stopwords."""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word.isalpha() and word not in stop_words]
    return tokens

# Apply cleaning to every statement
debate_text['tokens'] = debate_text['Text'].apply(clean_text)

print("Original text:")
print(debate_text['Text'].iloc[0][:100], "...")
print()
print("After cleaning (first 10 tokens):")
print(debate_text['tokens'].iloc[0][:10])

---
## Activity 2: Presidential Debates — TF-IDF by Party & Bigrams (~20 min)

*Your editor loved the word clouds! Now she asks: "But what's **different** between the parties? I don't just want the most common words — I want to know what each side is **uniquely** focused on. And check for any two-word phrases that stand out."*

---

### Demo: Group-Level TF-IDF

To find what's **unique** to each party, we:
1. Combine all text from each party into one big "document"
2. Run TF-IDF on those two documents
3. Words with high TF-IDF for Democrats are words Democrats use a lot but Republicans don't

In [ ]:
# Rejoin tokens into text strings
debate_text['joined_text'] = debate_text['tokens'].apply(lambda x: ' '.join(x))

# Split by party
democrats = debate_text[debate_text['Party'] == 'Democratic']
republicans = debate_text[debate_text['Party'] == 'Republican']

# Create one "document" per party
party_docs = {
    'Democratic': ' '.join(democrats['joined_text']),
    'Republican': ' '.join(republicans['joined_text'])
}

# Compute TF-IDF
vectorizer = TfidfVectorizer(max_features=1000)
tfidf_matrix = vectorizer.fit_transform(party_docs.values())

# Convert to long-format DataFrame
words = vectorizer.get_feature_names_out()
party_names = list(party_docs.keys())

tfidf_df = pd.DataFrame(tfidf_matrix.T.toarray(), index=words, columns=party_names).reset_index()
tfidf_df = tfidf_df.rename(columns={'index': 'word'})
tfidf_long = tfidf_df.melt(id_vars='word', var_name='Party', value_name='tfidf')

print("TF-IDF scores (sample):")
print(tfidf_long.sort_values('tfidf', ascending=False).head(10))

### Demo: Side-by-Side TF-IDF Bar Charts

In [ ]:
# Get top 15 TF-IDF words per party
top_tfidf = (
    tfidf_long
    .sort_values(by='tfidf', ascending=False)
    .groupby('Party')
    .head(15)
)

# Plot side by side
parties = top_tfidf['Party'].unique()
fig, axes = plt.subplots(ncols=2, figsize=(16, 6), sharey=False)

for ax, party in zip(axes, parties):
    data = top_tfidf[top_tfidf['Party'] == party].sort_values(by='tfidf')
    color = 'steelblue' if party == 'Democratic' else 'coral'
    sns.barplot(data=data, x='tfidf', y='word', ax=ax, color=color)
    ax.set_title(f"Top 15 TF-IDF Words \u2013 {party}", fontsize=13)
    ax.set_xlabel("TF-IDF Score")
    ax.set_ylabel("Word")

plt.tight_layout()
plt.show()

print("\nNotice: these are NOT the most common words overall.")
print("They are the words that make each party's language UNIQUE.")

### 📝 Activity 2a: Interpret the TF-IDF Results

Look at the TF-IDF bar charts above and answer these questions:

1. **Which 5 words have the highest TF-IDF for Democrats?**
2. **Which 5 words have the highest TF-IDF for Republicans?**
3. **What do these words tell you about each party's priorities?**
4. **Why is "people" NOT in either party's top words, even though it's the most common word overall?**

In [ ]:
# Activity 2a: Write your answers here as comments

# 1. Top 5 TF-IDF words for Democrats: ___people, think, know, country, going____________
# 2. Top 5 TF-IDF words for Republicans: __people, going, know, dont, im_____________
# 3. What do these suggest about priorities? ___People are the main priority for both parties____________
# 4. Why isn't "people" in the top TF-IDF words? ____People is non-distinctive, and TF-IDF charts pull the more unique words___________



### Demo: Extracting Bigrams

A **bigram** is a pair of consecutive words. "climate change" is more informative than just "climate" or "change" alone.

NLTK's `bigrams()` function creates these pairs from a list of tokens.

In [ ]:
# Generate bigrams from each statement's tokens
debate_text['bigrams'] = debate_text['tokens'].apply(lambda x: list(bigrams(x)))

# Flatten all bigrams into one list
all_bigrams = [bg for bigram_list in debate_text['bigrams'] for bg in bigram_list]

# Count frequencies
bigram_freq = Counter(all_bigrams)

# Convert to DataFrame
bigram_count_df = pd.DataFrame(bigram_freq.most_common(50), columns=['Bigram', 'Count'])

print("Top 10 bigrams:")
print(bigram_count_df.head(10))

### 📝 Activity 2b: Visualize the Top 20 Bigrams

Create a **bar chart** of the top 20 most frequent bigrams.

Steps:
1. Take the top 20 from `bigram_count_df`
2. Convert bigram tuples to strings: `.apply(lambda x: ' '.join(x))`
3. Plot with `sns.barplot()`

*Hint: the bigram column contains tuples like `('climate', 'change')` — you need to join them into a string for plotting.*

In [ ]:
# Activity 2b: Bar chart of top 20 bigrams
# Expected output: a horizontal bar chart of the 20 most frequent bigrams
# Hint: take top 20 from bigram_count_df,
#       convert tuples to strings with .apply(lambda x: " ".join(x)),
#       then plot with sns.barplot

# YOUR CODE HERE

top20_bigrams = bigram_count_df.head(20).copy()
top20_bigrams['Bigram'] = top20_bigrams['Bigram'].apply(lambda x: ' '.join(x))
plt.figure(figsize=(10, 6))
sns.barplot(data=top20_bigrams, y='Bigram', x='Count', color='steelblue')
plt.title('Top 20 Most Frequent Bigrams', fontsize=14)
plt.xlabel('Frequency')
plt.ylabel('Bigram')
plt.tight_layout()
plt.show()


### 📝 Activity 2c: Bigram Word Cloud

Create a **word cloud** from the top 100 bigrams.

Steps:
1. Create a dictionary of bigram strings and their counts
2. Generate a `WordCloud` from frequencies
3. Display with `plt.imshow()`

*Hint: you need to join the bigram tuples into strings for the dictionary keys.*

In [ ]:
# Activity 2c: Bigram word cloud
# Expected output: a word cloud where larger phrases appeared more often
# Hint: create a dict mapping bigram strings to counts,
#       then use WordCloud(...).generate_from_frequencies(dict)

# YOUR CODE HERE
top_bigrams = bigram_count_df.head(100).copy()
top_bigrams['Bigram'] = top_bigrams['Bigram'].apply(lambda x: ' '.join(x))
bigram_freq_dict = dict(zip(top_bigrams['Bigram'], top_bigrams['Count']))
wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(bigram_freq_dict)

plt.figure(figsize=(10, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Top 100 Bigrams', fontsize=16)
plt.show()



---
# 🏠 AFTER-CLASS ACTIVITIES

Complete these on your own after lecture (~30 min total).

---

## Activity 3: Bigram TF-IDF by Party (~15 min)

Instead of looking at single-word TF-IDF, let's find which **two-word phrases** are most distinctive for each party.

The trick: use `TfidfVectorizer(ngram_range=(2, 2))` to tell sklearn to only look at bigrams.

### Demo: Bigram TF-IDF Setup

The only difference from word-level TF-IDF is the `ngram_range` parameter:
- `ngram_range=(1, 1)` = single words (default)
- `ngram_range=(2, 2)` = bigrams only
- `ngram_range=(1, 2)` = both single words and bigrams

In [ ]:
# Demo: how ngram_range works
example_docs = ["the climate change debate is important",
                "tax reform helps the economy"]

# Bigram vectorizer
bigram_vec = TfidfVectorizer(ngram_range=(2, 2))
result = bigram_vec.fit_transform(example_docs)

print("Bigram features:")
print(bigram_vec.get_feature_names_out())

### 📝 Activity 3a: Compute Bigram TF-IDF by Party

Use the same `party_docs` dictionary from Activity 2, but this time fit a `TfidfVectorizer` with `ngram_range=(2, 2)` and `max_features=500`.

Then create side-by-side bar charts of the top 15 bigrams per party.

*Hint: the code is almost identical to the word-level TF-IDF demo — just change the vectorizer.*

In [ ]:
# Activity 3a: Bigram TF-IDF by party
# Expected output: side-by-side bar charts of top 15 bigrams per party
# Hint: use TfidfVectorizer(ngram_range=(2, 2), max_features=500)
#       then follow the same pattern as the word-level TF-IDF demo

# YOUR CODE HERE
debate_text['joined_text'] = debate_text['tokens'].apply(lambda x: ' '.join(x))

# Split by party
democrats = debate_text[debate_text['Party'] == 'Democratic']
republicans = debate_text[debate_text['Party'] == 'Republican']

# Create one "document" per party (this was already done in Activity 2 demo, so party_docs should be available)
# If not, recreate it here:
party_docs = {
    'Democratic': ' '.join(democrats['joined_text']),
    'Republican': ' '.join(republicans['joined_text'])
}

vectorizer = TfidfVectorizer(ngram_range=(2, 2), max_features=500)
tfidf_matrix = vectorizer.fit_transform(list(party_docs.values())) # Corrected: fit on party-level documents

bigrams = vectorizer.get_feature_names_out()
party_names = list(party_docs.keys())

tfidf_df = pd.DataFrame(tfidf_matrix.T.toarray(), index=bigrams, columns=party_names).reset_index()
tfidf_df = tfidf_df.rename(columns={'index': 'bigram'})
tfidf_long = tfidf_df.melt(id_vars='bigram', var_name='Party', value_name='tfidf')

# Get top 15 TF-IDF bigrams per party for plotting
top_tfidf_bigrams = (
    tfidf_long
    .sort_values(by='tfidf', ascending=False)
    .groupby('Party')
    .head(15)
)

# Plot side by side
parties = top_tfidf_bigrams['Party'].unique()
fig, axes = plt.subplots(ncols=2, figsize=(16, 6), sharey=False)

for ax, party in zip(axes, parties):
    data = top_tfidf_bigrams[top_tfidf_bigrams['Party'] == party].sort_values(by='tfidf')
    color = 'steelblue' if party == 'Democratic' else 'coral'
    sns.barplot(data=data, x='tfidf', y='bigram', ax=ax, color=color)
    ax.set_title(f"Top 15 Bigram TF-IDF \u2013 {party}", fontsize=13)
    ax.set_xlabel("TF-IDF Score")
    ax.set_ylabel("Bigram")

plt.tight_layout()
plt.show()


# Reflection:
# Top 3 bigrams for Democrats: _______________
# Top 3 bigrams for Republicans: _______________

---
## Activity 4: TF-IDF Word Clouds by Party (~15 min)

Create **TF-IDF-weighted word clouds** for each party. Instead of sizing words by raw frequency, size them by TF-IDF score — so uniquely important words stand out more.

### 📝 Activity 4a: TF-IDF Word Cloud for Democrats

Steps:
1. Filter `tfidf_long` (from Activity 2) to Democrats only
2. Sort by TF-IDF descending and take the top 100 words
3. Convert to a dictionary: `dict(zip(df['word'], df['tfidf']))`
4. Generate a `WordCloud` from that dictionary

*Hint: this is similar to the frequency word cloud, but you use TF-IDF scores instead of counts.*

In [ ]:
# Activity 4a: TF-IDF word cloud for Democrats
# Expected output: a word cloud where uniquely Democratic words are largest
# Hint: filter tfidf_long to Democrats, take top 100,
#       convert to dict, generate WordCloud

# YOUR CODE HERE

dem_tfidf = tfidf_long[tfidf_long['Party'] == 'Democratic']
dem_tfidf_top100 = dem_tfidf.sort_values(by='tfidf', ascending=False).head(100)
dem_tfidf_dict = dict(zip(dem_tfidf_top100['bigram'], dem_tfidf_top100['tfidf']))
wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(dem_tfidf_dict)
plt.figure(figsize=(10, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Top 100 Democratic Words', fontsize=16)
plt.show()


### 📝 Activity 4b: TF-IDF Word Cloud for Republicans

Repeat Activity 4a for Republicans.

In [ ]:
# Activity 4b: TF-IDF word cloud for Republicans
# Expected output: a word cloud where uniquely Republican words are largest
# Hint: same as 4a but filter to Republicans

# YOUR CODE HERE
rep_tfidf = tfidf_long[tfidf_long['Party'] == 'Republican']
rep_tfidf_top100 = rep_tfidf.sort_values(by='tfidf', ascending=False).head(100)
rep_tfidf_dict = dict(zip(rep_tfidf_top100['bigram'], rep_tfidf_top100['tfidf']))
wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(rep_tfidf_dict)
plt.figure(figsize=(10, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Top 100 Republican Words', fontsize=16)
plt.show()

# Reflection:
# How do TF-IDF word clouds differ from frequency word clouds?
# Answer: _____these are less filtered than TF-IDF since that uses more unique words and this word cloud does not__________


---
## Summary

Here's what you practiced in this notebook:

| Activity | Topic | Key Methods |
|----------|-------|-------------|
| 1a | Top words by character (bar charts) | `Counter`, `sns.barplot()` |
| 1b | Word clouds by character | `WordCloud`, `generate_from_frequencies()` |
| 1c | TF-IDF: Romeo vs. Juliet | `TfidfVectorizer`, side-by-side bar charts |
| 2a | Interpret TF-IDF results | Read bar charts, explain TF-IDF |
| 2b | Top bigrams (bar chart) | `bigrams()`, `Counter`, `sns.barplot()` |
| 2c | Bigram word cloud | `WordCloud` with bigram frequencies |
| 3a | Bigram TF-IDF by party | `TfidfVectorizer(ngram_range=(2,2))` |
| 4a-b | TF-IDF word clouds | `WordCloud` with TF-IDF scores |

**Key takeaways:**
- **Word counts** tell you what's talked about most overall
- **TF-IDF** tells you what makes each group's language unique
- **Bigrams** capture meaningful phrases that single words miss
- **Word clouds** are great for presentations; **bar charts** are better for precision
- The same pipeline works for single words and bigrams — just change `ngram_range`

---
## Reminders

- **Submit** this notebook via Canvas by the deadline.
- **Office hours**: Check the syllabus for times and location.
- **Project**: Make sure you are making progress on your unstructured data project. Reach out if you have questions!